In [ ]:
import numpy as np
import pandas as pd
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split



In [19]:
transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [20]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        image = Image.open(img_path).convert('RGB')
        label = row['diagnosis']
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
full_data = BlindnessDataset(
    csv_file='data/raw/aptos2019-blindness-detection/train.csv',
    img_dir='data/raw/aptos2019-blindness-detection/train_images',
    transform=transform
)

train_data, val_data = random_split(full_data, [3000, 662])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

In [22]:
image, label = train_data[0]
image.size()


torch.Size([3, 224, 224])

In [23]:
class_names = ['0 - No DR', '1 - Mild', '2 - Moderate', '3 - Severe', '4 - Proliferative DR']

In [24]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        #Convolutional layer 1
        self.conv1 = nn.Conv2d(3, 16, 5) #1st: # of input channels, 2nd: # of feature maps filters, 3rd: filter size (5 by 5) (224 - 5 / 1) + 1 = 220, so (16, 220, 220)
        
        #Pooling
        self.pool = nn.AvgPool2d(2,2) #Picked Avg

        #Convolutional layer 2
        self.conv2 = nn.Conv2d(16, 32, 5) # (32, 106, 106) => 110 - 5 + 1 = 106
        
        #Flatten
        self.fc1 = nn.Linear(32 * 26 * 26, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 5)

    def forward(self, x):
        #F.relu is just Max(0, x), never negative
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(x) 
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


        

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # should print "cuda"

net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
# optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
optimizer = optim.Adam(net.parameters(), lr=0.001)


print(device)
print(next(net.parameters()).device)

cuda
cuda
cuda:0


In [28]:
import time

In [ ]:

for epoch in range(5):
    print(f'Training epoch/round {epoch}...')
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()


    print(f'Epoch {epoch} done | Avg Loss: {running_loss/len(train_loader):.4f}')

Training epoch/round 0...
Epoch 0 | Batch 0/115 | Loss: 1.0715
Epoch 0 | Batch 10/115 | Loss: 1.0960
Epoch 0 | Batch 20/115 | Loss: 1.0726
Epoch 0 | Batch 30/115 | Loss: 1.1717
Epoch 0 | Batch 40/115 | Loss: 1.3315
Epoch 0 | Batch 50/115 | Loss: 1.0366
Epoch 0 | Batch 60/115 | Loss: 1.3675
Epoch 0 | Batch 70/115 | Loss: 1.2186
Epoch 0 | Batch 80/115 | Loss: 0.7457
Epoch 0 | Batch 90/115 | Loss: 0.9253
Epoch 0 | Batch 100/115 | Loss: 0.8641
Epoch 0 | Batch 110/115 | Loss: 0.8486
Epoch 0 done | Avg Loss: 1.0809 | Time: 272.6s
Training epoch/round 1...
Epoch 1 | Batch 0/115 | Loss: 0.6306
Epoch 1 | Batch 10/115 | Loss: 1.0609
Epoch 1 | Batch 20/115 | Loss: 1.0355
Epoch 1 | Batch 30/115 | Loss: 0.6949
Epoch 1 | Batch 40/115 | Loss: 1.1524
Epoch 1 | Batch 50/115 | Loss: 1.0827
Epoch 1 | Batch 60/115 | Loss: 0.6462
Epoch 1 | Batch 70/115 | Loss: 0.8933
Epoch 1 | Batch 80/115 | Loss: 0.6860
Epoch 1 | Batch 90/115 | Loss: 0.8778
Epoch 1 | Batch 100/115 | Loss: 0.8672
Epoch 1 | Batch 110/115 | 

In [33]:
torch.save(net.state_dict(), 'ash_trained_net.pth')

In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # should print "cuda"

net = NeuralNet().to(device)

net.load_state_dict(torch.load('ash_trained_net.pth'))

cuda


C:\Users\ashwa\AppData\Local\Temp\ipykernel_44100\550649100.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load('ash_trained_net.pth'))


<All keys matched successfully>

In [ ]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in val_loader:                          
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = net(images)                      
        _, predicted = torch.max(outputs, 1)         

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')


KeyError: 'diagnosis'